In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import skimage.io
import os 
import tqdm
import glob
import tensorflow 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import tensorflow as tf
from tensorflow.keras.utils import image_dataset_from_directory
from tensorflow.data.experimental import AUTOTUNE
from tensorflow.keras import Sequential, Input, Model
from tensorflow.keras.layers import RandomRotation, RandomZoom
from tensorflow.keras.layers.experimental.preprocessing import Rescaling
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras import applications
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers import SGD

from tqdm import tqdm
from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split
from skimage.io import imread, imshow
from skimage.transform import resize

from tensorflow.keras.models import Sequential
from tensorflow.keras.metrics import Precision, AUC,Recall
from tensorflow.keras.layers import InputLayer, BatchNormalization, Dropout, Flatten, Dense, Activation, MaxPool2D, Conv2D
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.applications.densenet import DenseNet169
import copy
import warnings
warnings.filterwarnings('ignore')
import tensorflow as tf
import keras
from keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import load_img, img_to_array
import matplotlib
import matplotlib.pylab as plt
from sklearn.utils import shuffle
from sklearn.metrics import confusion_matrix
from keras.applications.vgg16 import VGG16,preprocess_input
from keras.applications.vgg19 import VGG19,preprocess_input
from tensorflow.keras.utils import image_dataset_from_directory
#import tensorflow_addons as tfa
#from tensorflow_addons.optimizers import RectifiedAdam
#from tensorflow_addons.optimizers import AdamW

labels = pd.read_csv('AlzheimersFLAIR/patient_labels.csv')
labels

In [ ]:
from keras.preprocessing.image import ImageDataGenerator

BATCH_SIZE=32

SEED=1345

train_datagen=ImageDataGenerator(rescale=1./255,
                                shear_range=0,
                                zoom_range=0.2, rotation_range = 30)

validation_datagen=ImageDataGenerator(rescale=1./255)
test_datagen=ImageDataGenerator(rescale=1./255)


#Defining directories for train,validation,test 
train_dir = 'Splitted2D-19/train'
validation_dir = 'Splitted2D-19/val'
test_dir = 'Splitted2D-19/test'


#Defining generatores for train,validation,test

train_generator=train_datagen.flow_from_directory(
    train_dir,
    target_size=(224, 224),
    shuffle=True,
    seed = SEED,
    batch_size= 32,
    class_mode ='categorical',
)

validation_generator = validation_datagen.flow_from_directory(
    validation_dir,
    target_size=(224, 224),
    seed = SEED,
    shuffle=True,
    batch_size= 32,
    class_mode ='categorical',
)

# Define generator for test set using flow_from_directory
test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(224, 224),
    shuffle=True,
    seed = SEED,
    batch_size = 32,
    class_mode ='categorical',
)

In [ ]:
from keras.backend import sigmoid
def swish(x, beta = 1.3):
    return (x * sigmoid(beta * x))

from keras.utils import get_custom_objects
from keras.layers import Activation
get_custom_objects().update({'swish': Activation(swish)})

In [ ]:
from tensorflow.keras.optimizers.legacy import SGD, Adam, Nadam

input_shape = (224,224, 3)

#Create an instance of the VGG19 model
vgg19 = VGG19(include_top=False, input_shape=input_shape,
                   weights='imagenet')

# Freeze the layers of the VGG19 model
for layer in vgg19.layers:
    layer.trainable = False
    
# Create a new model with additional layers

#tf.keras.layers.LeakyReLU(0.1)

global_average_layer = tf.keras.layers.GlobalAveragePooling2D()
#init = tf.keras.initializers.HeNormal(seed=42)
prediction_layer = keras.layers.Dense(4,activation='softmax')

model = tf.keras.Sequential([vgg19, global_average_layer,
    keras.layers.BatchNormalization(),  
    keras.layers.Dropout(0.5),
    keras.layers.Dense(2048, activation='swish', kernel_initializer = 'glorot_uniform'),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(512, activation='swish', kernel_initializer = 'glorot_uniform'),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(256, activation='swish', kernel_initializer = 'glorot_uniform'),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(64, activation='swish', kernel_initializer = 'glorot_uniform'),
    keras.layers.Dropout(0.3),
    prediction_layer
])

model.compile(optimizer=SGD(learning_rate = 0.018, decay = 0.0015, momentum = 0.91, nesterov = True), 
              loss='categorical_crossentropy', metrics=['accuracy',tf.keras.metrics.AUC(),
                        tf.keras.metrics.Precision(),
                        tf.keras.metrics.Recall(),])